## Bonus Task – Monthly Summary Reports (Aggregations)

This notebook generates **monthly summary reports** based on the **latest cleaned dataset**
produced by the daily pipeline.

### Scope (What this notebook does)
- Reads the latest cleaned snapshot:
  `clean/vehicles_latest_clean.csv`
- Generates aggregated statistics for reporting purposes only

### Out of Scope (What this notebook does NOT do)
- No data cleansing or normalization is performed here  
- All data cleaning and standardization are handled in the **daily pipeline notebook**

### Monthly Reports Generated
The following CSV reports are created once per month:
- **Vehicles count by manufacturing year** (`by_year`)
- **Vehicles count by make** (`by_make`)
- **Vehicles count by model** (`by_model`)
- **Average vehicle age by ownership type** (`avg_age_by_type`)

### Output Location
All reports are saved under:
`dbfs:/Volumes/workspace/default/gov_data/reports/monthly/`

### Execution
This notebook is designed to run as a **monthly Databricks Job**.


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.gov_data;


In [0]:
import datetime
from pyspark.sql import functions as F

BASE_DIR_VOL = "dbfs:/Volumes/workspace/default/gov_data"

def vol_path(*parts: str) -> str:
    return "/".join([BASE_DIR_VOL.rstrip("/")] + [p.strip("/") for p in parts])

def ensure_dir(path: str):
    try:
        dbutils.fs.ls(path)
    except Exception:
        dbutils.fs.mkdirs(path)

PLATE_COLUMN = "mispar_rechev"

MONTHLY_DIR = vol_path("reports", "monthly")
ensure_dir(MONTHLY_DIR)

latest = vol_path("clean", "vehicles_latest_clean.csv")
print("Using latest:", latest)


In [0]:
def monthly_summary() -> str:
    df = spark.read.option("header", "true").csv(latest)
    print("Columns:", df.columns)
    print("Row count (sample):", df.limit(1).count())

    # ---- Cleansing בסיסי ----
    # מורידים כפילויות לפי מספר רישוי (אם יש)
    if PLATE_COLUMN in df.columns:
        df = df.dropDuplicates([PLATE_COLUMN])

    # שנה כ-int (אם קיימת)
    if "shnat_yitzur" in df.columns:
        df = df.withColumn("shnat_yitzur_int", F.col("shnat_yitzur").cast("int"))

    # גיל רכב ממוצע (אם אפשר)
    current_year = datetime.date.today().year
    if "shnat_yitzur_int" in df.columns:
        df = df.withColumn("vehicle_age", F.lit(current_year) - F.col("shnat_yitzur_int"))

    # ---- Aggregations ----
    outputs = []

    if "shnat_yitzur_int" in df.columns:
        by_year = (df.groupBy("shnat_yitzur_int").count()
                     .withColumnRenamed("count", "vehicles_count")
                     .orderBy(F.col("shnat_yitzur_int").desc()))
        outputs.append(("by_year", by_year))

    if "tozeret_nm" in df.columns:
        by_make = (df.groupBy("tozeret_nm").count()
                     .withColumnRenamed("count", "vehicles_count")
                     .orderBy(F.col("vehicles_count").desc()))
        outputs.append(("by_make", by_make))

    if "degem_nm" in df.columns:
        by_model = (df.groupBy("degem_nm").count()
                      .withColumnRenamed("count", "vehicles_count")
                      .orderBy(F.col("vehicles_count").desc()))
        outputs.append(("by_model", by_model))

    # גיל ממוצע (שורה אחת)
    if "vehicle_age" in df.columns and "baalut_norm" in df.columns:
        avg_age_by_type = (df.groupBy("baalut_norm")
                         .agg(F.avg("vehicle_age").alias("avg_vehicle_age_years"))
                         .orderBy("baalut_norm"))
        outputs.append(("avg_age_by_type", avg_age_by_type))

    # ---- כתיבה לקבצים (כל output לקובץ נפרד, הכי קריא) ----
    ym = datetime.date.today().strftime("%Y-%m")
    saved = []

    for name, sdf in outputs:
        tmp_out  = f"{MONTHLY_DIR}/_tmp_monthly_{ym}_{name}"
        out_path = f"{MONTHLY_DIR}/monthly_{ym}_{name}.csv"

        try:
            dbutils.fs.rm(tmp_out, recurse=True)
        except Exception:
            pass

        (sdf.coalesce(1)
            .write.mode("overwrite")
            .option("header", "true")
            .csv(tmp_out))

        part_files = [f.path for f in dbutils.fs.ls(tmp_out) if f.name.startswith("part-") and f.name.endswith(".csv")]
        if not part_files:
            raise RuntimeError(f"No part file produced for {name}")

        try:
            dbutils.fs.rm(out_path)
        except Exception:
            pass

        dbutils.fs.mv(part_files[0], out_path)
        dbutils.fs.rm(tmp_out, recurse=True)
        saved.append(out_path)

    print("Saved monthly files:")
    for p in saved:
        print(" -", p)

    return saved[0] if saved else None

monthly_summary()
